# DDRI Station-Day Baseline Modeling

이 노트북은 `station-day` 베이스라인 데이터셋을 사용해 기본 회귀 모델을 학습하고 평가한다.

- 튜닝용 학습 구간: `2023-01-01 ~ 2023-12-31`
- validation 구간: `2024-01-01 ~ 2024-12-31`
- 최종 테스트 구간: `2025-01-01 ~ 2025-12-31`
- 비교 모델: `LinearRegression`, `LightGBMRegressor`
- 평가 지표: `RMSE`, `MAE`, `R2`

검증 전략은 강한 계절성과 날씨 영향을 고려해 `연 단위 time-based split`을 사용한다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
DATA_DIR = ROOT / 'works/03_prediction/02_data'
IMAGE_DIR = ROOT / 'works/03_prediction/03_images'

TRAIN_PATH = DATA_DIR / 'ddri_station_day_train_baseline_dataset.csv'
TEST_PATH = DATA_DIR / 'ddri_station_day_test_main_eval_dataset.csv'


## 1. 데이터 로드

기본 베이스라인 데이터셋을 읽고 날짜형으로 변환한다.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])

train.shape, test.shape

## 2. 검증 전략과 피처 정책

`따릉이` 수요는 계절성과 날씨 영향이 크기 때문에 `2024 Q4` 같은 부분 분기만 validation으로 두면 대표성이 약할 수 있다.

따라서 이번 베이스라인은 아래 전략을 사용한다.

- `2023` : 모델 학습 및 1차 튜닝
- `2024` : 연 단위 validation
- `2025` : 최종 테스트

또한 같은 날짜에 관측된 `return_count`, `same_station_return_count`, `same_station_return_ratio`, `net_flow`는 예측 시점에 알 수 없기 때문에 피처에서 제외한다.

대신 아래 조합으로 baseline을 구성한다.

- 시간/공휴일 feature
- 일 단위 날씨 feature
- 정적 스테이션 정보
- `cluster_label`
- 과거 수요 기반 `lag/rolling` feature


In [ ]:
combined = pd.concat(
    [train.assign(dataset='train'), test.assign(dataset='test')],
    ignore_index=True,
)
combined = combined.sort_values(['station_id', 'date']).reset_index(drop=True)

grouped_target = combined.groupby('station_id')['rental_count']
combined['rental_count_lag1'] = grouped_target.shift(1)
combined['rental_count_lag7'] = grouped_target.shift(7)
combined['rolling_mean_7'] = grouped_target.shift(1).rolling(7).mean()
combined['rolling_std_7'] = grouped_target.shift(1).rolling(7).std()

train_model = combined[combined['dataset'] == 'train'].copy()
test_model = combined[combined['dataset'] == 'test'].copy()

FEATURE_COLUMNS = [
    'station_id', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'holiday_count',
    'is_holiday', 'is_business_holiday', 'temperature_mean', 'temperature_min',
    'temperature_max', 'humidity_mean', 'wind_speed_mean', 'precipitation_sum',
    'is_rainy_day', 'cluster_label', '위도', '경도',
    'rental_count_lag1', 'rental_count_lag7', 'rolling_mean_7', 'rolling_std_7'
]
CATEGORICAL_COLUMNS = ['station_id', 'cluster_label']
NUMERIC_COLUMNS = [col for col in FEATURE_COLUMNS if col not in CATEGORICAL_COLUMNS]

train_model[FEATURE_COLUMNS].head(3)

## 3. 연 단위 분할

`2023`으로 학습하고 `2024`를 validation으로 사용한다.
최종 테스트는 선택 모델을 `2023+2024` 전체로 다시 학습한 뒤 `2025`에서 평가한다.

In [ ]:
train_2023 = train_model[train_model['date'] < '2024-01-01'].copy()
valid_2024 = train_model[train_model['date'] >= '2024-01-01'].copy()
full_train = train_model.copy()

X_train = train_2023[FEATURE_COLUMNS].copy()
y_train = train_2023['rental_count'].copy()

X_valid = valid_2024[FEATURE_COLUMNS].copy()
y_valid = valid_2024['rental_count'].copy()

X_full = full_train[FEATURE_COLUMNS].copy()
y_full = full_train['rental_count'].copy()

X_test = test_model[FEATURE_COLUMNS].copy()
y_test = test_model['rental_count'].copy()

train_2023.shape, valid_2024.shape, full_train.shape, test_model.shape

## 4. 모델 정의

- `LinearRegression`: 해석이 쉬운 기준선 모델
- `LightGBMRegressor`: 비선형 관계를 빠르게 반영하는 트리 기반 모델


In [ ]:
linear_model = Pipeline(
    steps=[
        (
            'preprocess',
            ColumnTransformer(
                transformers=[
                    (
                        'categorical',
                        Pipeline(
                            steps=[
                                ('imputer', SimpleImputer(strategy='most_frequent')),
                                ('encoder', OneHotEncoder(handle_unknown='ignore')),
                            ]
                        ),
                        CATEGORICAL_COLUMNS,
                    ),
                    (
                        'numeric',
                        Pipeline(
                            steps=[
                                ('imputer', SimpleImputer(strategy='median')),
                                ('scaler', StandardScaler()),
                            ]
                        ),
                        NUMERIC_COLUMNS,
                    ),
                ]
            ),
        ),
        ('model', LinearRegression()),
    ]
)

lightgbm_model = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)


## 5. Validation 비교와 최종 테스트

먼저 `2023 → 2024 validation`으로 모델을 비교한다.
이후 선택 모델은 `2023+2024` 전체로 재학습해서 `2025`를 테스트한다.

In [ ]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
    }


def to_categorical_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['station_id'] = pd.Categorical(out['station_id'])
    out['cluster_label'] = pd.Categorical(out['cluster_label'])
    return out


results = []

linear_model.fit(X_train, y_train)
results.append(evaluate_predictions('LinearRegression', 'validation_2024', y_valid, linear_model.predict(X_valid)))
linear_model.fit(X_full, y_full)
results.append(evaluate_predictions('LinearRegression', 'test_2025_refit', y_test, linear_model.predict(X_test)))

lightgbm_model.fit(to_categorical_frame(X_train), y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM', 'validation_2024', y_valid, lightgbm_model.predict(to_categorical_frame(X_valid))))
lightgbm_model.fit(to_categorical_frame(X_full), y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM', 'test_2025_refit', y_test, lightgbm_model.predict(to_categorical_frame(X_test))))

results_df = pd.DataFrame(results)
results_df

## 6. 결과 저장

비교표와 최종 LightGBM feature importance 차트를 저장한다.

In [ ]:
metrics_path = DATA_DIR / 'ddri_station_day_baseline_model_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')

importance_df = pd.DataFrame(
    {
        'feature': FEATURE_COLUMNS,
        'importance': lightgbm_model.feature_importances_,
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

importance_path = DATA_DIR / 'ddri_station_day_lightgbm_feature_importance.csv'
importance_df.to_csv(importance_path, index=False, encoding='utf-8-sig')

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='importance', y='feature', hue='feature', legend=False, palette='Blues_r')
plt.title('DDRI Station-Day LightGBM Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
feature_image_path = IMAGE_DIR / 'ddri_station_day_lightgbm_feature_importance.png'
plt.savefig(feature_image_path, dpi=150)
plt.close()

metrics_path, importance_path, feature_image_path

## 7. 해석 메모

- `LinearRegression`은 기준선 모델로 사용한다.
- `LightGBM`은 `2024 validation`과 `2025 test` 모두에서 더 좋은 성능을 보이면 다음 단계 기본 모델 후보로 채택한다.
- 이번 실험의 핵심 포인트는 계절성이 강한 시계열 특성을 반영해 `2023/2024/2025` 연 단위 분할을 사용했다는 점이다.
- 또한 같은 날 운영 지표를 제외하고도 `lag + 날씨 + 공휴일 + 정적정보`만으로 baseline 성능을 확인했다.
